# Column Transformer with Mixed Types

This compares native Moxie with scikit-learn's
[mixed-types gallery example](https://scikit-learn.org/dev/auto_examples/compose/plot_column_transformer_mixed_types.html)
at revision `4358542a1bd39af2e7e1cbace8b9fed067473c5b`.

A single `ColumnTransformer` dispatches Titanic's numeric columns to median
imputation and standard scaling. It sends categorical columns to dense
one-hot encoding and chi-squared feature selection. Logistic regression fits
the resulting dense matrix.

The notebook defaults to seeded Titanic-like data for offline runs. Set
`MOXIE_TITANIC_OPENML=1` to use the gallery's [OpenML Titanic dataset
(version 1)](https://www.openml.org/d/40945). The
passenger classes 1, 2 and 3 are represented as string labels on both sides
to match the native table's string categorical storage. The target is Float64
for the categorical branch's chi-squared scorer and Int64 for classification.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("compose/plot_column_transformer_mixed_types")

## scikit-learn reference

In [2]:
import os

import numpy as np
import pandas as pd

use_openml = os.environ.get("MOXIE_TITANIC_OPENML") == "1"
if use_openml:
    from sklearn.datasets import fetch_openml

    X, y = fetch_openml(
        data_id=40945, data_home=ROOT / "builds/titanic-reference",
        as_frame=True, return_X_y=True,
    )
    assert X.shape == (1309, 13)
    X = X[["pclass", "sex", "embarked", "age", "fare"]].copy()
    X["pclass"] = pd.Categorical(X["pclass"].astype(str))
    y = y.astype(np.int64)
else:
    rng = np.random.RandomState(0)
    n = 1309
    pclass = rng.choice([1, 2, 3], size=n, p=[0.25, 0.2, 0.55])
    sex = rng.choice(["female", "male"], size=n, p=[0.36, 0.64])
    embarked = rng.choice(["C", "Q", "S"], size=n, p=[0.2, 0.1, 0.7])
    age = np.round(rng.normal(30 - 3 * (pclass - 2), 13, size=n).clip(0.5, 80), 1)
    fare = np.round(rng.lognormal(3.6 - 0.6 * (pclass - 1), 0.6, size=n), 2)
    age[rng.rand(n) < 0.2] = np.nan
    fare[rng.rand(n) < 0.01] = np.nan
    logit = 1.2 * (sex == "female") - 0.8 * (pclass - 2) - 0.02 * np.nan_to_num(age - 30) + 0.3 * (embarked == "C") - 0.6
    survived = (rng.rand(n) < 1 / (1 + np.exp(-2 * logit))).astype(int)
    X = pd.DataFrame({"pclass": pd.Categorical(pclass.astype(str)), "sex": pd.Categorical(sex),
                      "embarked": pd.Categorical(embarked), "age": age, "fare": fare})
    y = pd.Series(survived, name="survived")
if use_openml:
    print("OpenML Titanic (40945)")
X.head()

,pclass,sex,embarked,age,fare
0,3,male,S,13.8,18.66
1,3,male,S,27.6,18.32
2,3,female,C,NaN,15.93
3,3,male,S,17.4,9.90
4,2,female,S,54.7,25.02


In [3]:
import runpy

import sklearn.compose
import sklearn.feature_selection
import sklearn.impute
from sklearn.model_selection import train_test_split
from compose_parity_io import load_pinned, pinned_code, verify_reference

# Use the locked sklearn wheel for supporting imports and the pinned source
# checkout for the code under comparison.
sources = {
    "compose": "sklearn/compose/_column_transformer.py",
    "pipeline": "sklearn/pipeline.py",
    "impute": "sklearn/impute/_base.py",
    "scaler": "sklearn/preprocessing/_data.py",
    "encoders": "sklearn/preprocessing/_encoders.py",
    "encode": "sklearn/utils/_encode.py",
    "selection": "sklearn/feature_selection/_univariate_selection.py",
    "example": "examples/compose/plot_column_transformer_mixed_types.py",
}
verify_reference(sources.values())
load_pinned(sources["encode"], register=True)
pipeline = load_pinned(sources["pipeline"], register=True)
compose = load_pinned(sources["compose"])
impute = load_pinned(sources["impute"])
scaler = load_pinned(sources["scaler"])
encoders = load_pinned(sources["encoders"])
selection = load_pinned(sources["selection"])
for method, key in [
    (compose.ColumnTransformer.fit_transform, "compose"),
    (pipeline.Pipeline.fit_transform, "pipeline"),
    (impute.SimpleImputer.fit, "impute"),
    (scaler.StandardScaler.fit, "scaler"),
    (encoders.OneHotEncoder.fit, "encoders"),
    (selection.chi2, "selection"),
    (selection.SelectPercentile._get_support_mask, "selection"),
]:
    pinned_code(method, sources[key])

logistic_oracle = runpy.run_path(str(
    ROOT / "families/linear_model/algorithms/logistic/tests/parity.py"
))
LogisticRegression = logistic_oracle["oracle"]()

numeric = ["age", "fare"]
categorical = ["embarked", "sex", "pclass"]
X_train, X_test, y_train, y_test = train_test_split(
    X[categorical + numeric], y, test_size=0.2, random_state=0
)
# Exercise named routing when transform receives a different column order.
X_test = X_test[numeric + categorical[::-1]]
number_selector = compose.make_column_selector(dtype_include="number")
category_selector = compose.make_column_selector(dtype_include="category")


def preprocessor(num_columns, cat_columns):
    return compose.ColumnTransformer([
        ("num", pipeline.make_pipeline(
            impute.SimpleImputer(strategy="median"), scaler.StandardScaler()
        ), num_columns),
        ("cat", pipeline.make_pipeline(
            encoders.OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            selection.SelectPercentile(selection.chi2, percentile=50),
        ), cat_columns),
    ])


reference = preprocessor(number_selector, category_selector)
sk_train = reference.fit_transform(X_train, y_train)
sk_test = reference.transform(X_test)
sk_names = reference.get_feature_names_out()
named_reference = preprocessor(numeric, categorical)
named_train, named_test = X_train, X_test
if use_openml:
    # The gallery's named route passes integer passenger classes to the encoder.
    named_train = X_train.assign(pclass=X_train["pclass"].astype(np.int64))
    named_test = X_test.assign(pclass=X_test["pclass"].astype(np.int64))
np.testing.assert_array_equal(named_reference.fit_transform(named_train, y_train), sk_train)
np.testing.assert_array_equal(named_reference.transform(named_test), sk_test)
np.testing.assert_array_equal(named_reference.get_feature_names_out(), sk_names)

sk_classifier = LogisticRegression(tol=1e-9).fit(sk_train, y_train)
nb.share(y_train=y_train, y_test=y_test)
for split, frame in [("train", X_train), ("test", X_test)]:
    nb.share(**{f"{split}_numeric": frame[numeric].to_numpy()})
    for name in categorical:
        labels = frame[name].astype(object)
        nb.share(**{f"{split}_{name}": labels.fillna("").to_numpy(),
                    f"{split}_{name}_missing": labels.isna().to_numpy()})

scikit-learn source SHA256 {'sklearn/linear_model/_linear_loss.py': 'db6fd7e23b4619a76a46b8225876646328930db42d42e6af76223f72674afa3e', 'sklearn/linear_model/_logistic.py': '30863a8c7be82a63e2994405cf3fb29a4ab069eccb74ea6720582b4d8dec0b99', 'sklearn/linear_model/_glm/_newton_solver.py': 'f9c7caf323700d69d52ff709ea5e82d5c5fcfaa433f216cadaffce81621f6eae'}, support wheel 1.9.1


## Moxie

Each branch is a native transformer pipeline. `ColumnTable` owns the numeric
matrix and typed categorical columns. The notebook passes the transformed
matrices between Mojo cells to keep compilation within the memory cap.

In [4]:
%%mojo mixed_preprocessor
from moxie._categorical import categorical_column
from moxie.compose import ColumnEntry, ColumnTable, ColumnTransformer, make_column_selector
from moxie.feature_selection import BuiltinScorer, SelectPercentile
from moxie.impute import SimpleImputer
from moxie.pipeline import make_pipeline
from moxie.preprocessing import OneHotEncoder, StandardScaler
from moxie_nb_io import load, load_strings, save, save_ints, save_strings


def load_table(split: String, reverse_categories: Bool) raises -> ColumnTable[]:
    var table = ColumnTable(load(split + "_numeric"), ["age", "fare"])
    var categorical: List[String] = ["embarked", "sex", "pclass"]
    if reverse_categories:
        categorical = ["pclass", "sex", "embarked"]
    for name in categorical:
        var cells = List[Optional[String]]()
        var labels = load_strings(split + "_" + name)
        var missing = load[DType.bool](split + "_" + name + "_missing")
        for i in range(len(labels)):
            cells.append(None if missing.item(i) else Optional[String](labels[i]))
        table.add_categorical(name, categorical_column(cells))
    return table^


def main() raises:
    var train = load_table("train", False)
    var test = load_table("test", True)
    var number: List[String] = ["number"]
    var category: List[String] = ["category"]
    var num_selector = make_column_selector(dtype_include=number^)
    var cat_selector = make_column_selector(dtype_include=category^)
    save_strings("numeric", num_selector(train).names.copy())
    save_strings("categorical", cat_selector(train).names.copy())

    var num = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    var cat = make_pipeline(
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        SelectPercentile(BuiltinScorer("chi2"), percentile=50),
    )
    var preprocessor = ColumnTransformer(
        ColumnEntry("num", num^, num_selector^),
        ColumnEntry("cat", cat^, cat_selector^),
    )
    save("train", preprocessor.fit_transform(train, load("y_train")))
    save("test", preprocessor.transform(test))
    save_strings("names", preprocessor.get_feature_names_out())
    var spans = List[Int]()
    if preprocessor.output_indices_:
        ref indices = preprocessor.output_indices_.value()
        for name in ["num", "cat", "remainder"]:
            spans.append(indices[name].start)
            spans.append(indices[name].stop)
    save_ints("spans", spans)

In [5]:
mx = nb.results["mixed_preprocessor"]
nb.share(MX_train=mx["train"], MX_test=mx["test"])

In [6]:
%%mojo logistic_regression
from moxie.linear_model import LogisticRegression
from moxie_nb_io import load, save, save_floats


def main() raises:
    var train = load("MX_train")
    var test = load("MX_test")
    var model = LogisticRegression(tol=1e-9)
    _ = model.fit(train, load[DType.int64]("y_train"))
    save("pred", model.predict(test))
    save("proba", model.predict_proba(test))
    save_floats("accuracy", [model.score(test, load[DType.int64]("y_test"))])

## Parity

In [7]:
moxie_nb.side_by_side(sklearn=pd.DataFrame(sk_test[:5], columns=sk_names),
                      moxie=pd.DataFrame(mx["test"][:5], columns=mx["names"]))
sk_spans = [edge for name in ["num", "cat", "remainder"]
            for edge in (reference.output_indices_[name].start, reference.output_indices_[name].stop)]
nb.compare("Numeric dtype selector", number_selector(X_train), mx["numeric"], exact=True)
nb.compare("Categorical dtype selector", category_selector(X_train), mx["categorical"], exact=True)
nb.compare("get_feature_names_out", sk_names, mx["names"], exact=True)
nb.compare("output_indices_", sk_spans, mx["spans"], exact=True)
num_span = reference.output_indices_["num"]
cat_span = reference.output_indices_["cat"]
for split, expected in [("train", sk_train), ("test", sk_test)]:
    nb.compare(f"{split} numeric output", expected[:, num_span], mx[split][:, num_span],
               rtol=1e-12, atol=1e-12,
               reason="Float64 scaler reductions can sum in different orders.")
    nb.compare(f"{split} selected one-hot output", expected[:, cat_span],
               mx[split][:, cat_span], exact=True)

classified = nb.results["logistic_regression"]
nb.compare("LogisticRegression predictions", sk_classifier.predict(sk_test),
           classified["pred"], exact=True)
nb.compare("LogisticRegression probabilities", sk_classifier.predict_proba(sk_test),
           classified["proba"], rtol=1e-6, atol=1e-8,
           reason="L-BFGS line search and convergence can differ between implementations.")
nb.compare("LogisticRegression accuracy", [sk_classifier.score(sk_test, y_test)],
           classified["accuracy"], exact=True)
nb.report()
nb.assert_parity()

,num__age,num__fare,cat__sex_female,cat__sex_male,cat__pclass_1,cat__pclass_3
0,1.596599,-0.835759,0.0,1.0,0.0,1.0
1,-0.107641,-0.451979,1.0,0.0,0.0,0.0
2,0.349594,-0.258225,0.0,1.0,0.0,1.0
3,-0.440176,0.579664,0.0,1.0,1.0,0.0
4,-0.091014,0.551719,0.0,1.0,0.0,1.0
,num__age,num__fare,cat__sex_female,cat__sex_male,cat__pclass_1,cat__pclass_3
0,1.596599,-0.835759,0.0,1.0,0.0,1.0
1,-0.107641,-0.451979,1.0,0.0,0.0,0.0
2,0.349594,-0.258225,0.0,1.0,0.0,1.0
3,-0.440176,0.579664,0.0,1.0,1.0,0.0


check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
Numeric dtype selector,PASS,"(2,)","(2,)",,,exact,0 mismatches,
Categorical dtype selector,PASS,"(3,)","(3,)",,,exact,0 mismatches,
get_feature_names_out,PASS,"(6,)","(6,)",,,exact,0 mismatches,
output_indices_,PASS,"(6,)","(6,)",,,exact,0 mismatches,
train numeric output,PASS,"(1047, 2)","(1047, 2)",1.29e-14,2.25e-11,"rtol=1e-12, atol=1e-12",,Float64 scaler reductions can sum in different orders.
train selected one-hot output,PASS,"(1047, 4)","(1047, 4)",,,exact,0 mismatches,
test numeric output,PASS,"(262, 2)","(262, 2)",1.24e-14,1.24e-12,"rtol=1e-12, atol=1e-12",,Float64 scaler reductions can sum in different orders.
test selected one-hot output,PASS,"(262, 4)","(262, 4)",,,exact,0 mismatches,
LogisticRegression predictions,PASS,"(262,)","(262,)",,,exact,0 mismatches,
LogisticRegression probabilities,PASS,"(262, 2)","(262, 2)",4.11e-08,1.77e-07,"rtol=1e-06, atol=1e-08",,L-BFGS line search and convergence can differ between implementations.
